# Extract frozen GLIM EEG tokens [96, 1024] + Gate-1 collapse verdict

Route B, Gate 1 (D-027). Use a **GPU**. Attach the canonical sharded dataset
`thestonedape/task-aware-eegtotext` (v1) and the checkpoint `thestonedape/glim-zuco-checkpoint`;
enable Internet and the private secret `GITHUB_TOKEN`.

This extracts the 96 unpooled EEG tokens GLIM already emits (prompt-neutral
`all_masked`, identical to the forward pass that produced P4b's pooled vectors),
for the full primary ZuCo2 NR/TSR cohort (9,011 trials). It then runs the
token-collapse diagnostic and prints a **COLLAPSED / WEAK / RICH** verdict --
the make-or-break Gate-1 readout. The held-out test is never touched.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'd03ed05ca8a6d0bb72f5a741e9112ca75c17624a'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = 'e1f202cb793cfe7292fbc0072a4c26a7dd0660d9'
GLIM_WORKTREE = '/kaggle/working/GLIM'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
OUTPUT = '/kaggle/working/task-aware-eeg2text-glim-tokens'
BATCH_SIZE = 16
CHUNK_SIZE = 128
DTYPE = 'float16'
DIAGNOSTICS_SAMPLE = 1024
assert len(COMMIT) == len(GLIM_COMMIT) == 40
assert all(len(v) == 64 for v in (EXPECTED_INDEX_SHA256, CHECKPOINT_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, torch
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', GLIM_WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == GLIM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.0'], check=True)
env = os.environ.copy(); env['PYTHONPATH'] = WORKTREE
for t in ('evaluation.test_token_late_interaction', 'evaluation.test_token_diagnostics', 'evaluation.test_extract_frozen_glim_tokens'):
    subprocess.run([sys.executable, '-B', '-m', 'unittest', t], check=True, cwd=WORKTREE, env=env)
print({'clone': 'PASS', 'token_tests': 'PASS'})

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
checkpoint_paths = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert len(checkpoint_paths) == 1, ('Attach exactly one GLIM checkpoint', checkpoint_paths)
checkpoint = checkpoint_paths[0]
state = hashlib.sha256()
with open(checkpoint, 'rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        state.update(block)
assert state.hexdigest() == CHECKPOINT_SHA256, 'checkpoint SHA mismatch'
print({'dataset_root': dataset_root, 'checkpoint': checkpoint})

In [ ]:
def token_command(output, smoke_limit=None):
    cmd = [
        sys.executable, '-B', os.path.join(WORKTREE, 'evaluation', 'extract_frozen_glim_tokens.py'),
        '--dataset-root', dataset_root, '--output-root', output, '--glim-root', GLIM_WORKTREE,
        '--checkpoint', checkpoint, '--glim-commit', GLIM_COMMIT,
        '--expected-index-sha256', EXPECTED_INDEX_SHA256,
        '--expected-checkpoint-sha256', CHECKPOINT_SHA256,
        '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--chunk-size', str(CHUNK_SIZE),
        '--dtype', DTYPE, '--diagnostics-sample', str(DIAGNOSTICS_SAMPLE),
    ]
    if smoke_limit is not None:
        cmd += ['--smoke-limit', str(smoke_limit)]
    return cmd

# Pre-flight crash-check: 4 rows through the real GLIM forward pass (seconds).
smoke_out = '/kaggle/working/token-preflight'
if os.path.exists(smoke_out):
    shutil.rmtree(smoke_out)
subprocess.run(token_command(smoke_out, smoke_limit=4), check=True, cwd=WORKTREE,
               env={**os.environ, 'PYTHONPATH': WORKTREE})
shutil.rmtree(smoke_out)
print('PRE-FLIGHT: PASS -- proceeding to full extraction')

In [ ]:
# Full extraction of the primary ZuCo2 NR/TSR cohort (9,011 trials).
# Resumable: hash-valid chunks are reused; re-run this cell after any interruption.
subprocess.run(token_command(OUTPUT), check=True, cwd=WORKTREE,
               env={**os.environ, 'PYTHONPATH': WORKTREE})

In [ ]:
index = json.load(open(os.path.join(OUTPUT, 'token_index.json'), encoding='utf-8'))
assert index['token_shape'] == [96, 1024] and index['dtype'] == DTYPE
assert index['prompt_mode'] == 'all_masked' and index['glim_commit'] == GLIM_COMMIT
assert index['total_rows'] == 9011, index['total_rows']
# Re-hash every chunk against its recorded sha256.
for entry in index['chunks']:
    p = os.path.join(OUTPUT, entry['token_file'])
    state = hashlib.sha256()
    with open(p, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    assert state.hexdigest() == entry['sha256'], p
run_metadata = {
    'status': 'pass', 'project_commit': COMMIT, 'glim_commit': GLIM_COMMIT,
    'checkpoint_sha256': CHECKPOINT_SHA256, 'dataset_index_sha256': EXPECTED_INDEX_SHA256,
    'total_rows': index['total_rows'], 'token_shape': index['token_shape'], 'dtype': DTYPE,
    'combined_chunk_sha256': index['combined_chunk_sha256'],
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True); handle.write('\n')
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
print({'rows': index['total_rows'], 'chunks': index['num_chunks'],
       'combined_chunk_sha256': index['combined_chunk_sha256']})
print('GLIM TOKEN EXTRACTION: PASS')
print('The Gate-1 VERDICT was printed by the extraction cell above'
      ' (COLLAPSED / WEAK / RICH). Record it in the token spec.')

After PASS, save the printed OUTPUT (`task-aware-eeg2text-glim-tokens`) as a new
**private** Kaggle dataset, Version 1. The Gate-1 verdict decides the next move:
RICH -> proceed to Gate 2 (freeze the pooled-vs-token design); COLLAPSED/WEAK ->
fall back to Route A. Do not build the training loop before the verdict is in.